In [112]:
import torch
import torch.nn as nn
import mmap
import random
import pickle
from torch.nn import functional as F

device = 'cuda' if torch.cuda.is_available else 'cpu'
print(device)

block_size = 40
batch_size = 128
max_iters = 1000
eval_iter = 200
learning_rate = 3e-4 # other common lr 3e-3, 1e-3, 3e-4
n_embd = 576
n_layer = 12
dropout = 0.2
n_head = 12

cuda


In [113]:
with open('OpenWebText/vocab.txt', 'r', encoding='utf-8') as f:
    text = f.read()
    chars = sorted(list(set(text)))
vocab_size = len(chars)

In [114]:
string_to_int = {ch:i for i, ch in enumerate(chars)}
int_to_string = {i:ch for i, ch in enumerate(chars)}
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

In [115]:
# Using mmap to get small snippet of code from large file without opening them (which use an enormous amount of memory)
def get_random_chunk(split):
    filename = 'OpenWebText/train_split.txt' if split == 'train' else 'OpenWebText/val_split.txt'
    with open(filename, 'rb') as f:
        with mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_READ) as mm:
            # Determine the file size and decide the starting point of reading
            file_size = len(mm)
            start_pos = random.randint(0, file_size - block_size * batch_size)

            mm.seek(start_pos)
            block = mm.read(block_size * batch_size - 1)

            decoded_block = block.decode('utf-8', errors = 'ignore').replace('\r', '')

            data = torch.tensor(encode(decoded_block), dtype=torch.long)
    return data

In [116]:
def get_batch(split):
    data = get_random_chunk(split)
    ix = torch.randint(len(data) - block_size, (batch_size,))
    # print(ix)
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [117]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iter)
        for k in range(eval_iter):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [118]:
# Single head of self-attention (masked)
class Head(nn.Module):

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        
        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        v = self.value(x)
        out = wei @ v
        return out

In [119]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(num_heads * head_size, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

In [120]:
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
                nn.Linear(n_embd, 4 * n_embd),
                nn.ReLU(),
                nn.Linear(4 * n_embd, n_embd),
                nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

In [121]:
# Transformer decoder block
class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        y = self.sa(x)
        x = self.ln1(x + y)
        y = self.ffwd(x)
        x = self.ln2(x + y)
        return x

In [122]:
class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        # nn.Embedding is used to create position embedding when creating a specialized GPT
        # The standard sinusoidal functions (sin(pos/10000^(2i/d)) and cos(pos/10000^(2i/d))) are used to train base models
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        # Create n_layer number of layers/decoders in sequence (one depends on another)
        # -> different from module-list in MultiHeadAttention class because it computes heads completely independently (parallel)
        self.block = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])

        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

        self.apply(self._init_weights)

    # Initialize the weight
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
        
    def forward(self, index, targets = None):
        B, T = index.shape

        tok_emb = self.token_embedding_table(index)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.block(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        
        if targets == None:
            loss = None
        else:
            # B = batch size, T = sequence size (how many tokens in one sequence), C = vocab size
            B, T, C = logits.shape
            # Needs to reshape so that each column represents a single score, instead of a sequence of scores
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, index, max_new_tokens):
        # index is the current context of shape(B, C)
        for _ in range(max_new_tokens):
            # Get the prediction
            index_cond = index[:, -block_size:]
            logits, loss = self.forward(index_cond)
            # Get the last sequence
            logits = logits[:, -1, :]
            # Get the probability distribution of the next token
            probs = F.softmax(logits, dim=-1)
            # Using the PD to get the next token
            index_next = torch.multinomial(probs, num_samples=1)
            # Concatenate the newly generated token to the original token
            index = torch.cat((index, index_next), dim=1)
        return index

model = GPTLanguageModel(vocab_size)

# loads the saved model
with open('model-01.pkl', 'rb') as f:
    model = pickle.load(f)

m = model.to(device)

In [123]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
# Training loop (common structure)
for iter in range(max_iters):
    if iter % eval_iter == 0:
        losses = estimate_loss()
        print(f'step: {iter}, loss: {losses}')

        
    xb, yb = get_batch('train');
    logits, loss = model.forward(xb, yb);
    # zero_grad should be on only when training on large dataset that requires the model to remember previous content
    optimizer.zero_grad(set_to_none=True);
    # correct the model and store the correction in grad
    loss.backward();
    # update the parameters using the new grad
    optimizer.step();

print(loss.item())

step: 0, loss: {'train': tensor(1.4293), 'val': tensor(1.4253)}
step: 200, loss: {'train': tensor(1.4274), 'val': tensor(1.4228)}
step: 400, loss: {'train': tensor(1.4493), 'val': tensor(1.4454)}
step: 600, loss: {'train': tensor(1.4170), 'val': tensor(1.4209)}
step: 800, loss: {'train': tensor(1.4494), 'val': tensor(1.4256)}
1.403594970703125


In [124]:
# Saving the learned parameters (the trained model essentially) to a file named model-01.pkl
with open('model-01.pkl', 'wb') as f:
    pickle.dump(model, f)